In [11]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import requests
from pathlib import Path
from bs4 import BeautifulSoup
from dotenv import load_dotenv


load_dotenv()

root = os.environ['ROOT_PATH']
root = Path(root).resolve()

sys.path.append(str(root))

from data.ib import IBAPI

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
# pd.read_csv(root + '/data/hudsonthames/sp500_constituents-detailed.csv')
pd.read_csv(str(root) + '/data/equities/ibex35_constituents_2023-10-31.csv')

,Ticker,Empresa,Sede,Entrada,Sector[30]​,ISIN,Ponderación (Sep. 2022)
0,ANA,Acciona,Alcobendas,2015,Construcción,ES0125220311,112
1,ANE,Acciona Energía,Alcobendas,2022,Energías renovables,ES0105563003,36
2,ACX,Acerinox,Madrid,2015,"Mineral, metales y transformación",ES0132105018,49
3,ACS,ACS,Madrid,1998,Construcción,ES0167050915,203
4,AENA,Aena,Madrid,2015,Transporte y distribución,ES0105046009,351
5,AMS,Amadeus IT Group,Madrid,2011,Electrónica y software,ES0109067019,519
6,MTS,ArcelorMittal,Ciudad de Luxemburgo,2009,"Mineral, metales y transformación",LU1598757687,76
7,SAB,Banco Sabadell,Sabadell,2004,Bancos y cajas de ahorro,ES0113860A34,141
8,BKT,bankinter.,Madrid,1992,Bancos y cajas de ahorro,ES0113679I37,115
9,BBVA,BBVA,Bilbao,1992,Bancos y cajas de ahorro,ES0113211835,948


## Fetch Constituents

In [23]:
wiki_eng = "https://en.wikipedia.org/wiki/IBEX_35"
wiki_esp = "https://es.wikipedia.org/wiki/IBEX_35"

headers = { "User-Agent": "Mozilla/5.0" }

response = requests.get(wiki_esp, headers = headers)

tables = pd.read_html(response.text)

ibex35_constituents = pd.DataFrame()

for table in tables:
    if 'Ticker' in table.columns and 'ISIN' in table.columns:
        ibex35_constituents = table.copy()

ibex35_constituents

ibex35_constituents.to_csv(root + '/data/equities/ibex35_constituents_2023-10-31.csv', index = False)

/var/folders/bc/hnzwjdn546lcc572zg36k_vc0000gn/T/ipykernel_25903/3784388869.py:8: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


## Fetch Prices

In [16]:
ibapi = IBAPI()

# TODO: fetch using ISIN
tickers = ['SAB1', 'ACS']

ibapi.get_multiple(tickers, currency = 'EUR', csv_path = str(root) + "/data/equities")

ERROR -1 2104 Market data farm connection is OK:cafarm
ERROR -1 2104 Market data farm connection is OK:hfarm
ERROR -1 2104 Market data farm connection is OK:eufarmnj
ERROR -1 2104 Market data farm connection is OK:cashfarm
ERROR -1 2104 Market data farm connection is OK:usfuture
ERROR -1 2104 Market data farm connection is OK:jfarm
ERROR -1 2104 Market data farm connection is OK:usfarm.nj
ERROR -1 2104 Market data farm connection is OK:eufarm
ERROR -1 2104 Market data farm connection is OK:usfarm
ERROR -1 2106 HMDS data farm connection is OK:euhmds
ERROR -1 2106 HMDS data farm connection is OK:cashhmds
ERROR -1 2106 HMDS data farm connection is OK:fundfarm
ERROR -1 2106 HMDS data farm connection is OK:ushmds
ERROR -1 2158 Sec-def data farm connection is OK:secdefnj
ERROR 1 2174 Warning: You submitted request with date-time attributes without explicit time zone. Please switch to use yyyymmdd-hh:mm:ss in UTC or use instrument time zone, like US/Eastern. Implied time zone functionality wi

[IB] SAB1: 254 bars fetched.


ERROR 1 2174 Warning: You submitted request with date-time attributes without explicit time zone. Please switch to use yyyymmdd-hh:mm:ss in UTC or use instrument time zone, like US/Eastern. Implied time zone functionality will be removed in the next API release


[IB] ACS: 254 bars fetched.


,SAB1,ACS
datetime,,
20250404,-0.107377,-0.053920
20250407,-0.057984,-0.058205
20250408,0.029858,0.038197
20250409,-0.012935,-0.018603
20250410,0.067329,0.043387
...,...,...
20260327,-0.004584,-0.017208
20260330,0.000329,-0.001946
20260331,0.002631,0.021442


In [31]:
ALPHA_VANTAGE_KEY = "OES1IQGDSQCVOVQV"

import requests
import pandas as pd
from io import StringIO

# API_KEY = "YOUR_ALPHA_VANTAGE_KEY"

def fetch_daily_csv(symbol: str) -> pd.DataFrame | None:
    url = (
        "https://www.alphavantage.co/query"
        f"?function=TIME_SERIES_DAILY"
        f"&symbol={symbol}"
        f"&datatype=csv"
        f"&apikey={ALPHA_VANTAGE_KEY}"
    )
    r = requests.get(url, timeout=30)
    text = r.text.strip()

    # Alpha Vantage often returns an error message as plain text/JSON-like text
    if not text or "Error Message" in text or "Note" in text or "Information" in text:
        print(f"Failed for {symbol}: {text[:200]}")
        return None

    try:
        df = pd.read_csv(StringIO(text))
    except Exception as e:
        print(f"CSV parse failed for {symbol}: {e}")
        return None

    if "timestamp" not in df.columns or "close" not in df.columns:
        print(f"No daily data columns for {symbol}")
        return None

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["return"] = df["close"].pct_change()
    return df

# Try a few candidate symbol formats for one stock
candidates = ["SAB.BME", "SAB.ES"]

for sym in candidates:
    df = fetch_daily_csv(sym)
    if df is not None and not df.empty:
        print(f"Success with {sym}")
        print(df.tail())
        break

Failed for SAB.BME: {
    "Error Message": "Invalid API call. Please retry or visit the documentation (https://www.alphavantage.co/documentation/) for TIME_SERIES_DAILY."
}
Failed for SAB.ES: {
    "Information": "Thank you for using Alpha Vantage! Please consider spreading out your free API requests more sparingly (1 request per second). You may subscribe to any of the premium plans at ht


In [51]:
import requests

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
# url = f'https://www.alphavantage.co/query?function=SYMBOL_SEARCH&keywords=SAN&apikey={ALPHA_VANTAGE_KEY}'
url = 'https://eodhd.com/api/eod/SAB.MAD?api_token=demo&fmt=json'
r = requests.get(url)
data = r.json()
data[-1]
# for i in data['bestMatches']:
#     print(i['4. region'])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [24]:
from datetime import datetime
now = datetime.now()
date_time = now.strftime('%Y%m%d-%H:%M:%S')
date_time

'20260330-19:00:23'

In [ ]:
I can use ib script to do so

,AAPL,MSFT,AMZN,META
Date,,,,
2015-07-24,-0.005273,-0.003687,0.097972,0.015821
2015-07-27,-0.013896,-0.012843,0.003759,-0.028675
2015-07-28,0.004969,-0.000220,-0.010124,0.011893
2015-07-29,-0.003161,0.020953,0.005646,0.017840
2015-07-30,-0.005041,0.012746,0.014669,-0.018352
...,...,...,...,...
2017-02-16,-0.001181,-0.000155,0.001709,0.002998
2017-02-17,0.002734,0.001550,0.001102,-0.002316
2017-02-21,0.007221,-0.002012,0.013455,0.001423
